<a href="https://colab.research.google.com/github/atanilson/Comp702/blob/main/Comp702_APP_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Installing the necessary libraries

In [1]:
%%capture
!pip -q install rasterio
!pip install jupyter_bokeh

In [ ]:
# Standard imports
import os
import io
import pandas as pd
import numpy as np
from PIL import Image

# Geospacial processing packages
import geopandas as gpd

#import shapely
import rasterio as rio
import rasterio.mask
from shapely.geometry import box

# Mapping and plotting libraries
import matplotlib.pyplot as plt
import matplotlib.colors as cl
import folium

# Model
import torch
from torchvision import datasets, models, transforms

import panel as pn
pn.extension()

device = "cuda" if torch.cuda.is_available() else "cpu"

def generate_tiles(image_file, size=64):
    """Genarates size*size polygon tiles.
    """

    # Open the raser image using rasterio
    raster = rio.open(image_file)
    width, height = raster.shape

    # Create a dictionary which will conatin our 64Z64 px polygon tiles
    # Later convert this dict into GeoPandas DataFrame
    geo_dict = {"id":[],"geometry":[]}
    index = 0

    # Do a sliding windows across the raste image
    for w in range(0, width, size):
      for h in range(0, height, size):
          # Create a Window of your disired size
          window = rio.windows.Window(h, w, size, size)

          # Get the georeferenced windowss bounds
          bbox = rio.windows.bounds(window, raster.transform)

          # Create a shapely geometry from the bounding box
          bbox = box(*bbox)

          # Create a unique id for each geometry
          uid = str(index)

          # Update dictionary
          geo_dict["id"].append(uid)
          geo_dict["geometry"].append(bbox)

          index += 1

    # Cast dictionary as a GeoPandas DataFrame
    results = gpd.GeoDataFrame(pd.DataFrame(geo_dict))

    # Set CRS to EPSG: 4326
    results.set_crs("EPSG:4326", inplace=True)

    raster.close()

    return results



transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def predict_crop(image, shape, classes, model, show=False):
    """
    Generate prediction from a raster image using a geometry crop,
    """
    with rio.open(image) as img:
        # Crop source image using polygon shape
        out_image, out_transform = rio.mask.mask(img, shape, crop=True)

        # Crop out black (zero) border
        _, x_nonzero, y_nonzero = np.nonzero(out_image)
        out_image = out_image[
            :,
            np.min(x_nonzero):np.max(x_nonzero),
            np.min(y_nonzero):np.max(y_nonzero)
        ]

        # Convert to PIL image for model input
        np_img = np.moveaxis(out_image, 0, -1)  # (bands, H, W) -> (H, W, bands)

        pil_img = Image.fromarray(np_img.astype(np.uint8))

        # Apply transforms and make prediction
        input_tensor = transform(pil_img).to(device)
        output = model(input_tensor.unsqueeze(0))
        _, pred = torch.max(output, 1)
        label = str(classes[int(pred[0])])

        if show:
            pil_img.show(title=label)

        return label



file_input = pn.widgets.FileInput(accept='.tif')

message = pn.widgets.StaticText(name='Message', value='Nothing')

# Get file when imputed
def update_map(input_file=None):
  if input_file:
    message.value = "Uploaded"
    # Convert the uploaded binary data to a file-like object
    file_bytes = io.BytesIO(file_input.value)
    # Open with rasterio
    image = rio.open(file_bytes)

    message.value = "Creating tiles"
    tiles = generate_tiles(file_bytes, size=64)

    # LULC Classes
    classes = [
        "AnnualCrop",
        "Forest",
        "HerbaceousVegetation",
        "Highway",
        "Industrial",
        "Pasture",
        "PermanentCrop",
        "Residential",
        "River",
        "SeaLake"
    ]

    message.value = "Loading model"
    #Get the model
    path_drive = "RESNET50_0001_SDG-RGB_B32_WW_EP150_S224.pth"

    model_2 = models.resnet50()
    model_2.fc = torch.nn.Linear(in_features=model_2.fc.in_features, out_features=len(classes))
    model_2.load_state_dict(torch.load(f=path_drive, map_location=device))
    model_2 = model_2.to(device)

    model_2.eval()



    # Get label

    #
    message.value = "Making prediction: "
    labels = [] # Stor prediction
    #for index in tqdm(range(len(tiles)), total=len(tiles)):
    l = len(tiles)
    for index in range(l):
      message.value = f"Making prediction: {index}/{l}"
      try:
        # Use model_2 instead of model
        label = predict_crop(file_bytes, [tiles.iloc[index]['geometry']], classes, model_2)
        labels.append(label)
      except:
        labels.append("Error")
        continue

    # Create tile
    tiles['pred'] = labels


    message.value = "Creating map..."

    # We map each class to a corresponding color
    colors = {
      'AnnualCrop' : 'lightgreen',
      'Forest' : 'forestgreen',
      'HerbaceousVegetation' : 'yellowgreen',
      'Highway' : 'gray',
      'Industrial' : 'red',
      'Pasture' : 'mediumseagreen',
      'PermanentCrop' : 'chartreuse',
      'Residential' : 'magenta',
      'River' : 'dodgerblue',
      'SeaLake' : 'blue',
      'Error' : 'black'
    }
    tiles['color'] = tiles["pred"].apply(
      lambda x: cl.to_hex(colors.get(x))
    )
    # Divide Tiles





    # Instantiate map centered on the centroid
    map = folium.Map(zoom_start=10)


    # Add Google Satellite basemap
    folium.TileLayer(
          tiles = 'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
          attr = 'Google',
          name = 'Google Satellite',
          overlay = True,
          control = True
    ).add_to(map)

    # Add LULC Map with legend
    legend_txt = '<span style="color: {col};">{txt}</span>'
    for label, color in colors.items():

      # Specify the legend color
      name = legend_txt.format(txt=label, col=color)
      feat_group = folium.FeatureGroup(name=name)

      # Add GeoJSON to feature group
      subtiles = tiles[tiles.pred==label]
      if len(subtiles) > 0:
        folium.GeoJson(
            subtiles,
            style_function=lambda feature: {
              'fillColor': feature['properties']['color'],
              'color': 'black',
              'weight': 0,
              'fillOpacity': 0.5,
            },
            name='LULC Map'
        ).add_to(feat_group)
        map.add_child(feat_group)

    folium.LayerControl().add_to(map)
    # Reset zoom with data available
    map.fit_bounds(map.get_bounds())
    message.value = "Done!"
    return pn.pane.HTML(map._repr_html_(), width=900, height=900)
    # Classify
    # Process

    # Return map
"""
"""
# Attach callback
#file_input.param.watch(update_plot, 'value')
mapa= pn.bind(update_map, file_input)
# Layout
main_area=pn.Column(
    "# Upload GeoTIFF and View",
    file_input,
    mapa,
    message
)



MAIN=pn.WidgetBox(main_area)


tab2 = pn.GridSpec(width=1100, height=700, nrows=3, ncols=3)

tab2[0:3, 0:3] = MAIN  # Filters


title = "COMP702 - Landcover Decision Support Tool"
tab2.servable(title=title)


"""
title = "Panel Demo - Image Classification"
pn.template.BootstrapTemplate(
    title=title,
    main=main,
    main_max_width="min(50%, 698px)",
    header_background="#F08080",
).servable(title=title)
"""


In [7]:
pn.extension(sizing_mode="stretch_width")

layout1 = pn.Column(styles={"background": "green"}, sizing_mode="stretch_both")
layout2 = pn.Column(styles={"background": "red"}, sizing_mode="stretch_both")
layout3 = pn.Column(styles={"background": "blue"}, sizing_mode="stretch_both")

template = pn.template.FastGridTemplate(site="Panel", title="App", prevent_collision=True)
template.main[0:2,0:6]=layout1
template.main[0:2,6:12]=layout2
template.main[2:3,0:12]=layout3
template.servable()

FastGridTemplate
    [js_area] HTML(None, height=0, margin=0, sizing_mode='fixed', width=0)
    [actions] TemplateActions()
    [browser_info] BrowserInfo(dark_mode=False, device_pixel_ratio=1.5, language='en-US', timezone='Europe/London', timezone_offset=-60, webdriver=False, webgl=True)
    [busy_indicator] LoadingSpinner(height=20, width=20)
    [main-136090885097040] Column(sizing_mode='stretch_both', styles={'background': 'green'})
    [main-136090883253968] Column(sizing_mode='stretch_both', styles={'background': 'red'})
    [main-136090918767248] Column(sizing_mode='stretch_both', styles={'background': 'blue'})